In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Set a fixed random seed so the results are reproducible
np.random.seed(123)

# Create the dataset
n = 80  # number of samples (you can change this)
x = np.linspace(-3, 3, n).reshape(-1, 1)  # input feature: n points from -3 to 3
t = 0.5 * x.ravel()**2 + x.ravel()  # true quadratic relationship (no noise)
y = t + np.random.randn(n) * 0.6  # add noise to get the observed data

def plot_polynomial_fit(x, t, y, deg):
    """
    Fit a polynomial to the data and plot the original data,
    the fitted curve, and the ideal (noise-free) curve.

    Parameters
    ----------
    x : ndarray
        x-coordinates of the data
    y : ndarray
        y-coordinates of the data
    deg : int
        Degree of the polynomial
    np.polyfit(x, y, deg) :
        Least-squares polynomial fit; returns the polynomial coefficients
    np.poly1d(...) :
        Builds a polynomial function from the coefficients
    """
    # Flatten x to 1-D, as required by np.polyfit
    x_flat = x.ravel()
    # Fit a polynomial of the given degree
    p = np.poly1d(np.polyfit(x_flat, y, deg))


    # Plot original data (red dots), fitted curve (blue line),
    # and ideal curve (red dashed line)
    plt.plot(x, y, 'ro', label='Original Data')
    plt.plot(x, p(x), '-', label=f'Degree {deg} Fit')
    plt.plot(x, 0.5 * x_flat**2 + x_flat, 'r--', label='Ideal Result')

    # Show the legend
    plt.legend()

plt.figure(figsize=(18, 4), dpi=200)
degrees = [1, 3, 10]  # degrees of the polynomial
titles = ['Under Fitting', 'Fitting', 'Over Fitting']  # titles of the subplots
for index, deg in enumerate(degrees):
    plt.subplot(1, 3, index + 1)
    plot_polynomial_fit(x, t, y, deg)
    plt.title(titles[index], fontsize=20)

plt.show()


In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error


# Regularization limits model complexity. We use degree=10 (overfitting) as the baseline.
degree = 10
# PolynomialFeatures(include_bias=False) avoids duplicating the intercept of the linear model
models = {
    'No regularization': make_pipeline(
        PolynomialFeatures(degree, include_bias=False),
        LinearRegression()
    ),
    'L1 (Lasso)': make_pipeline(
        PolynomialFeatures(degree, include_bias=False),
        Lasso(alpha=0.01, max_iter=100000)
    ),
    'L2 (Ridge)': make_pipeline(
        PolynomialFeatures(degree, include_bias=False),
        Ridge(alpha=0.1)
    ),
    'ElasticNet': make_pipeline(
        PolynomialFeatures(degree, include_bias=False),
        ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=100000)
    )
}

plt. figure(figsize=(10, 6))  # create a new figure
# Prediction curve of each regularized model
for name, model in models.items():
    model.fit(x, y)  # train the model on the data

    y_pred = model.predict(x)  # predict on the training data

    # Training error and test error
    train_mse = mean_squared_error(y, y_pred)  # vs noisy data
    test_mse = mean_squared_error(t, y_pred)  # vs the ideal (noise-free) curve

    plt.plot(x, y_pred, '--', linewidth=2,
             label=f'{name} (train_MSE: {train_mse:.3f}, test_MSE: {test_mse:.3f})')
# Ideal (noise-free) curve for reference
plt.plot(x, 0.5 * x.ravel()**2 + x.ravel(), 'k-', linewidth=2, label='Ideal Result')

plt.xlabel('x')
plt.ylabel('y')
plt.title('Comparison of Different Regularization Methods under Overfitting (Degree=10)')
plt.legend(loc='best', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ==================== Dropout & Early Stopping ====================
# Dropout and early stopping are regularization methods for neural networks.
# They cannot be added to the sklearn Pipeline above, so we build an MLP
# with Keras that is large enough to overfit, and use it to show both methods.
import matplotlib.pyplot as plt  # plotting library
import tensorflow as tf  # deep learning framework (preinstalled in Colab)
from tensorflow.keras.models import Sequential  # sequential model: layers are added in order
from tensorflow.keras.layers import Dense, Dropout  # Dense = fully connected layer; Dropout = randomly drop neurons
from tensorflow.keras.callbacks import EarlyStopping  # stops training when validation loss stops improving
from sklearn.preprocessing import StandardScaler  # standardizes the data
from sklearn.metrics import mean_squared_error  # mean squared error, used to evaluate predictions

# Neural networks are sensitive to input scale, so standardize x first
# (mean = 0, standard deviation = 1) to help training converge faster.
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)  # compute mean/std from the data, then transform x
y_2d = y.reshape(-1, 1)  # Keras needs y as a 2-D array of shape (n_samples, 1)


def build_mlp(with_dropout, dropout_rate=0.3, seed=0):
    """Build a large MLP; if with_dropout=True, add Dropout layers between hidden layers."""
    tf.keras.utils.set_random_seed(seed)  # fix the random seed so all models start with the same weights
    model = Sequential()  # create a sequential model (layers are added in order)
    model.add(Dense(128, activation='relu', input_shape=(1,)))  # layer 1: 128 neurons, relu activation, input = 1 feature (x)
    if with_dropout:
        model.add(Dropout(dropout_rate))  # randomly drop 30% of neurons during training
    model.add(Dense(128, activation='relu'))  # layer 2: 128 neurons
    if with_dropout:
        model.add(Dropout(dropout_rate))
    model.add(Dense(1))  # output layer: 1 neuron, predicts y (regression problem)
    model.compile(optimizer='adam', loss='mse')  # optimizer: adam; loss: mean squared error
    return model


# Train four models with the same structure: a 2x2 comparison
# (with/without Dropout) x (with/without early stopping):
#   1) No Dropout + early stopping
#   2) Dropout(0.3) + early stopping
#   3) Dropout(0.3) + no early stopping
#   4) No Dropout + no early stopping (runs all 200 epochs), used as a baseline
epochs_dict = {}  # stores the actual number of epochs each model trained for
results = {}  # dict: model name -> (trained model, training history, use early stopping or not)
for name, use_dropout, use_early_stop in [
        ('No Dropout, Early Stopping', False, True),
        ('Dropout (0.3), Early Stopping', True, True),
        ('No Dropout, No Early Stopping', False, False),
        ('Dropout (0.3), No Early Stopping', True, False)]:
    model = build_mlp(with_dropout=use_dropout)  # build the model according to the config
    callbacks = []  # list of callbacks, empty for now
    if use_early_stop:
        # Stop if validation loss does not improve for 50 epochs; restore the best weights
        callbacks.append(EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True))
    # Train the model: at most 200 epochs; validation_split=0.2 keeps 20% of the data
    # as a validation set; batch_size=16 updates weights every 16 samples; verbose=0 hides progress.
    history = model.fit(x_scaled, y_2d, epochs=200, validation_split=0.2,
                        batch_size=16, callbacks=callbacks, verbose=0)
    epochs_dict[name] = len(history.history["loss"])  # record the actual number of epochs trained
    results[name] = (model, history, use_early_stop)


# Prediction curve comparison (same train/test MSE metrics as above;
# the four models show the effect of Dropout and early stopping).
plt.figure(figsize=(10, 6))  # create a 10x6 inch figure
for name, (model, _, _) in results.items():  # loop over the four models
    y_pred = model.predict(scaler.transform(x), verbose=0).ravel()  # standardize x, predict, flatten to 1-D
    train_mse = mean_squared_error(y, y_pred)  # training error: predictions vs noisy data y
    test_mse = mean_squared_error(t, y_pred)  # test error: predictions vs ideal curve t
    plt.plot(x, y_pred, '--', linewidth=2,  # plot the predicted curve
             label=f'{name}: trained {epochs_dict.get(name)} epchos (train_MSE: {train_mse:.3f}, test_MSE: {test_mse:.3f})')
plt.plot(x, t, 'k-', linewidth=2, label='Ideal Result')  # ideal (noise-free) curve as reference
plt.xlabel('x')
plt.ylabel('y')
plt.title('Dropout & Early Stopping Regularization (MLP)')
plt.legend(loc='best', fontsize=9)  # legend
plt.grid(True, alpha=0.3)  # grid lines
plt.tight_layout()  # adjust layout automatically
plt.show()  # show the figure
